# Estimación de peso de pollos

## Colegio de Posgraduados

### COA661 Inteligencia Artificial

Profesor: Dr. Juan Manuel González Camacho

Entrega: José Alfredo Martínez

En este notebook se entrenan 4 modelos para predecir el peso de los pollos: Support Vector Regressor, Random Forest Regressor, K Nearest Neighbors y Multilayer Perceptron.

Se utilizan las 8 características del Dataset

In [1]:
#Librerías
import numpy as np
import pandas as pd

import optuna

from IPython.display import clear_output

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import sklearn.model_selection

import joblib

## Código

In [2]:
# Cargar datos
datos1 = pd.read_excel('./misDatos.xlsx', sheet_name = 'X_train')
datos2 = pd.read_excel('./misDatos.xlsx', sheet_name = 'X_test')

fil, col = datos1.shape

X_train = datos1.iloc[:, 0 : col - 1].to_numpy()
Y_train = datos1.iloc[:, col - 1].to_numpy()

X_test = datos2.iloc[:, 0 : col - 1].to_numpy()
Y_test = datos2.iloc[:, col - 1].to_numpy()

In [3]:
# Cargar datos de estandarización
scx = joblib.load('T_scaler8.pkl')

In [4]:
# Estandarizar los datos
X_train_std = scx.transform(X_train)
X_test_std = scx.transform(X_test)

print(X_train_std.shape)

(2439, 8)


## Support Vector Regressor

In [5]:
# Optimización con Optuna
def objective(trial):
    clear_output(wait=True)
    
    kernel = trial.suggest_categorical('kernel', ['rbf'])
    C = trial.suggest_float('C', 0.1, 100, log = True)
    gamma = trial.suggest_float('gamma', 0.001, 10, log = True)
    epsilon = trial.suggest_float('epsilon', 0.0001, 1)

    modelo = SVR(kernel = kernel, C = C, gamma = gamma, epsilon = epsilon)
    # Crear el modelo y realizar la Cross Validation
    modelo = SVR(kernel = kernel, C = C, gamma = gamma, epsilon = epsilon)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_std, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:03:54,121] Trial 199 finished with value: 0.8154442809210554 and parameters: {'kernel': 'rbf', 'C': 4.520035255849773, 'gamma': 3.26086347211676, 'epsilon': 0.015146483588436638}. Best is trial 30 with value: 0.8181205440761078.


Cross Validation
Mejores hiperparámetros:  {'kernel': 'rbf', 'C': 2.7149932756991126, 'gamma': 4.118227283168486, 'epsilon': 0.0017752497394650092}
R2:  0.82


In [6]:
# Guardar datos de modelo
svr = SVR(**study.best_params)
joblib.dump(svr, './modelo_svr8.pkl')

['./modelo_svr8.pkl']

## Random Forest Regressor

In [7]:
# Optimización con Optuna
def objective(trial):
    clear_output(wait=True)
    
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    max_depth = trial.suggest_int('max_depth', 2, 35)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    modelo = RandomForestRegressor(n_estimators = n_estimators, max_depth = max_depth, min_samples_split = min_samples_split)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:15:20,349] Trial 199 finished with value: 0.8713475201256744 and parameters: {'n_estimators': 155, 'max_depth': 26, 'min_samples_split': 3}. Best is trial 158 with value: 0.8737917989637175.


Cross Validation
Mejores hiperparámetros:  {'n_estimators': 423, 'max_depth': 24, 'min_samples_split': 2}
R2:  0.87


In [8]:
# Guardar datos de modelo
rf = RandomForestRegressor(**study.best_params)
joblib.dump(rf, './modelo_rf8.pkl')

['./modelo_rf8.pkl']

## K Neighbors Regressor

In [9]:
def objective(trial):
    clear_output(wait=True)
    
    n_neighbors = trial.suggest_int('n_neighbors', 1, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan'])

    modelo = KNeighborsRegressor(n_neighbors = n_neighbors, weights = weights, metric = metric)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_std, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:15:26,770] Trial 199 finished with value: 0.8654473160110715 and parameters: {'n_neighbors': 12, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 16 with value: 0.8661327581563978.


Cross Validation
Mejores hiperparámetros:  {'n_neighbors': 9, 'weights': 'distance', 'metric': 'manhattan'}
R2:  0.87


In [10]:
# Guardar datos de modelo
knn = KNeighborsRegressor(**study.best_params)
joblib.dump(knn, './modelo_knn8.pkl')

['./modelo_knn8.pkl']

## Multilayer Perceptron Regressor

In [11]:
def objective(trial):
    clear_output(wait=True)

    hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes', [(25,), (50,), (25, 25), (50, 25)])
    activation = trial.suggest_categorical('activation', ['relu', 'tanh'])
    alpha = trial.suggest_float('alpha', 0.00001, 0.1, log = True)
    learning_rate_init = trial.suggest_float('learning_rate_init', 0.0001, 0.1, log = True)

    modelo = MLPRegressor(hidden_layer_sizes = hidden_layer_sizes, activation = activation, alpha = alpha, learning_rate_init = learning_rate_init, max_iter = 2000, random_state = 42)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_std, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()

    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:16:58,789] Trial 199 finished with value: 0.7297607289305958 and parameters: {'hidden_layer_sizes': (25, 25), 'activation': 'relu', 'alpha': 0.0792487924442378, 'learning_rate_init': 0.002955873723925447}. Best is trial 182 with value: 0.7477133187997389.


Cross Validation
Mejores hiperparámetros:  {'hidden_layer_sizes': (50, 25), 'activation': 'relu', 'alpha': 0.09852749887892169, 'learning_rate_init': 0.003084368869722724}
R2:  0.75


In [12]:
# Guardar datos de modelo
mlp = MLPRegressor(**study.best_params)
joblib.dump(mlp, './modelo_mlp8.pkl')

['./modelo_mlp8.pkl']